# Payment Failure Risk Classifier — Google Colab Training Notebook

This notebook trains a deterministic Risk Classification Model on payment failure features.
It outputs the trained model weights and `.keras` file to place into `backend/app/ml/models/risk_classifier.keras`.

In [ ]:
import numpy as np
import json
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# Generate 10,000 synthetic failure transactions
np.random.seed(42)
num_samples = 10000

amounts = np.clip(np.random.lognormal(mean=8.5, sigma=1.2, size=num_samples), 100, 200000)
amount_scaled = (amounts - 100) / (200000 - 100)
order_history = np.random.poisson(lam=3.5, size=num_samples)
dispute_count = np.random.binomial(n=3, p=0.05, size=num_samples)
velocity_score = np.random.beta(a=1.5, b=6.0, size=num_samples)
disposable_email = np.random.binomial(n=1, p=0.08, size=num_samples)
error_severity = np.random.choice([0.1, 0.25, 0.45, 0.7, 0.9], size=num_samples, p=[0.45, 0.25, 0.15, 0.10, 0.05])
retries = np.random.choice([0, 1, 2, 3], size=num_samples, p=[0.55, 0.25, 0.12, 0.08])
retry_ratio = retries / 3.0
lifetime_spend = order_history * np.random.uniform(500, 3000, size=num_samples)
lifetime_spend_scaled = np.clip(lifetime_spend / 50000.0, 0.0, 1.0)

X = np.column_stack([
    amount_scaled,
    np.clip(order_history / 15.0, 0.0, 1.0),
    np.clip(dispute_count / 3.0, 0.0, 1.0),
    velocity_score,
    disposable_email,
    error_severity,
    retry_ratio,
    lifetime_spend_scaled
])

logit = (
    2.2 * amount_scaled
    - 2.8 * (order_history / 15.0)
    + 3.5 * (dispute_count / 3.0)
    + 3.2 * velocity_score
    + 2.6 * disposable_email
    + 2.9 * error_severity
    + 1.8 * retry_ratio
    - 2.1 * lifetime_spend_scaled
    - 1.5
)
prob = 1.0 / (1.0 + np.exp(-logit))
y = (prob >= 0.50).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

In [ ]:
# Train Keras Neural Network
model = keras.Sequential([
    layers.Input(shape=(8,)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

history = model.fit(X_train, y_train, epochs=25, batch_size=64, validation_split=0.15)

# Evaluate on held-out test set
test_loss, test_acc, test_auc = model.evaluate(X_test, y_test)
print(f"Test ROC-AUC: {test_auc:.4f}, Test Accuracy: {test_acc:.4f}")

In [ ]:
# Save Model Artifacts
model.save("risk_classifier.keras")
print("Saved 'risk_classifier.keras'!")
print("Download this file and place it into: backend/app/ml/models/risk_classifier.keras in your project workspace.")